<div align="center">
    <h1>Carolina Rojas Duque</h1>
    <h2>Prueba técnica - Data Scientist Sumz</h2>
    <h3>16 de mayo de 2024</h3>
</div>


En este notebook compendié el procedimiento para la solución de la prueba técnica facilita en detalle. Encontrarán las etapas de:

-   Adquisición y limpieza de datos
-   Análisis exploratorio de datos
    -   Con énfasis en la solución de las preguntas planteadas
-   Ingeniería de características y generación de modelo predictivo
    -   TODO: poner diferentes approaches


In [351]:
import itertools

# evitar logging de información de librería en la que se basa Prophet
import logging

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import (
    plot_components_plotly,
    plot_plotly,
)
from statsmodels.regression.linear_model import OLS
from tqdm import tqdm

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

## Carga y preprocesamiento de datos


In [340]:
catalogo_productos = pd.read_csv(
    "./data/catalogo_productos.csv", index_col="id_producto"
)
demanda = pd.read_csv("./data/demanda.csv")
demanda_test = pd.read_csv("./data/demanda_test.csv")

catalogo_productos.head()

,categoria,subcategoria,tamaño,premium,marca_exclusiva,estacional,nit_proveedor
id_producto,,,,,,,
1,bebidas,Agua,Mediano,0.0,1,0,890900667.0
2,jabones,Antibacterial,Pequeño,NaN,0,0,890900667.0
3,bebidas,Gaseosa,Pequeño,0.0,1,0,890900667.0
4,congelados,Vegetales,Pequeño,0.0,1,1,890900667.0
5,shampoos,Acondicionador,Pequeño,0.0,1,0,890900667.0


In [341]:
catalogo_productos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 135 entries, 1 to 135
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   categoria        135 non-null    object 
 1   subcategoria     103 non-null    object 
 2   tamaño           132 non-null    object 
 3   premium          130 non-null    float64
 4   marca_exclusiva  135 non-null    int64  
 5   estacional       135 non-null    int64  
 6   nit_proveedor    134 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 8.4+ KB


Se identifica que **premium** tiene 5 valores vacíos, **tamaño** tiene 3, y **subcategoría** tiene 29. Verifiquemos a cuáles corresponden (esto es posible solo por el bajo número de datos faltantes, que se pueda revisar a mano).


In [342]:
catalogo_productos[catalogo_productos.isna().any(axis=1)]

,categoria,subcategoria,tamaño,premium,marca_exclusiva,estacional,nit_proveedor
id_producto,,,,,,,
2,jabones,Antibacterial,Pequeño,NaN,0,0,890900667.0
9,productos_enlatados_y_alimentos_envasados,NaN,Mediano,0.0,1,0,890900667.0
13,aseo,NaN,Pequeño,0.0,1,0,890900667.0
16,productos_lacteos,Leche,Mediano,NaN,0,0,890900667.0
21,productos_enlatados_y_alimentos_envasados,NaN,Pequeño,0.0,0,0,890900667.0
23,productos_lacteos,Yogur,NaN,0.0,1,0,890900667.0
31,carnes_y_aves,Pavo,NaN,0.0,0,0,890900667.0
33,productos_enlatados_y_alimentos_envasados,NaN,Pequeño,0.0,1,1,890900667.0
39,productos_enlatados_y_alimentos_envasados,NaN,Pequeño,0.0,1,0,890900667.0


No se observan comportamientos particulares para las filas faltantes. Por tanto, se decide completar la información de **premium** y **tamaño** con la moda de las variables dado que son pocos los valores faltantes y estos datos sintéticos no sesgan peligrosamente las distribuciones. Mientras que para el caso de la variable **subcategoría**, se decide crear una nueva con el valor de **Desconocido**, para evitar sesgar de formas inesperadas los procesos posteriores. En cuanto al nit faltante, también se llena con desconocido.


In [343]:
# Con base en eso, para las columnas premium y tamaño podemos reemplazar por la moda,
moda_premium = catalogo_productos["premium"].mode()[0]
moda_tamano = catalogo_productos["tamaño"].mode()[0]
catalogo_productos["premium"] = (
    catalogo_productos["premium"].fillna(moda_premium).astype(int)
)
catalogo_productos["tamaño"] = catalogo_productos["tamaño"].fillna(moda_tamano)

# y para subcategoria y nit_proveedor podemos reemplazar por un valor que indique que no se conoce
catalogo_productos["subcategoria"] = catalogo_productos["subcategoria"].fillna(
    "Desconocido"
)
catalogo_productos["nit_proveedor"] = catalogo_productos["nit_proveedor"].fillna(
    "Desconocido"
)

In [344]:
catalogo_productos.isna().sum()  # Verificamos que no haya valores nulos

categoria          0
subcategoria       0
tamaño             0
premium            0
marca_exclusiva    0
estacional         0
nit_proveedor      0
dtype: int64

Continuamos verificando los datasets con la demanda y demanda_test. Aquí no observamos valores faltantes en primera medida, solo se observa la necesidad de convertir las fechas a variable **datetime** para manipularlas fácilmente después.


In [345]:
print("---------- Demanda ----------")
print(demanda.head())
print(demanda.info())
demanda["date"] = pd.to_datetime(demanda["date"])

print("---------- Demanda test ----------")
print(demanda_test.head())
print(demanda_test.info())
demanda_test["date"] = pd.to_datetime(demanda_test["date"])

---------- Demanda ----------
         date  id_producto  demanda
0  2020-01-01            3      869
1  2020-01-01            4      407
2  2020-01-01            5      182
3  2020-01-01            6       91
4  2020-01-01            7      520
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126460 entries, 0 to 126459
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   date         126460 non-null  object
 1   id_producto  126460 non-null  int64 
 2   demanda      126460 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 2.9+ MB
None
---------- Demanda test ----------
         date  id_producto
0  2022-12-01            1
1  2022-12-01            2
2  2022-12-01            3
3  2022-12-01            4
4  2022-12-01            5
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11786 entries, 0 to 11785
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       -------------- 

Una vez realizada esta limpieza preliminar, entramos de lleno a verificar el comportamiento de la demanda para los diferentes productos, y su relación con las características de los mismos.


## Análisis exploratorio de datos


### Impacto de la tienda competidora en ventas


In [16]:
fig = px.line(
    demanda, x="date", y="demanda", color="id_producto", title="Demanda de productos"
)
fig.show()

-   Se observa el muestreo diario de los productos. No se verifica si hay días faltantes para los productos porque de antemano se ha decidido no completar esta información en caso de haberlos. Esto, justificado en que en dicho escenario primero convendría preguntarle al cliente o al equipo de ingeniería de datos, antes de hacer supuestos sobre dicha información.

-   Se observa además el comportamiento mayoritariamente oscilante de la demanda, presentando picos en los meses de junio-julio diciembre-enero, y valles en marzo-abril y octubre. Esto indica una periodicidad estimada visualmente de 6 meses en la data, información importante para cuando montemos los modelos predictivos.

-   También verificamos los muestreos de los productos de demanda estacional, que se venden los meses de noviembre, diciembre y enero de cada año.

-   Y finalmente empezamos a notar el impacto en las ventas para algunos de los productos en el momento de apertura de la tienda competidora. Se hace evidente también notar que dicho evento ocurre cerca de la mitad de las señales que tenemos, lo que resulta muy conveniente para reconocer que, incluso si hubiese un cambio en el valor constante alrededor del que ocurren las oscilaciones, estás siguen presentando el mismo comportamiento antes y después, lo que argumenta a favor de usar todo el dataset para la generación del modelo posterior.

Con estas observaciones en mente, resulta igual difícil verificar el comportamiento de todos los productos compendiados en una misma gráfica. Por lo que decido compendiar las demandas de los productos de las mismas categorías y subcategorías (sumándolos), y presentar las demandas por subcategoría para cada categoría dada.


In [17]:
demanda_merged = demanda.merge(
    catalogo_productos, left_on="id_producto", right_index=True
)
demanda_per_cat_subcat = (
    demanda_merged.groupby(["categoria", "subcategoria", "date"])["demanda"]
    .sum()
    .reset_index()
)

In [18]:
categories = demanda_per_cat_subcat["categoria"].unique()
categories_pretty = [cat.replace("_", " ").title() for cat in categories]

fig = make_subplots(rows=len(categories), cols=1, subplot_titles=categories_pretty)

for i, category in enumerate(categories):
    # tomar data solo de la categoría actual
    data = demanda_per_cat_subcat[
        demanda_per_cat_subcat["categoria"] == category
    ].sort_values("date")
    subcategories = data["subcategoria"].unique()

    for subcategory in subcategories:
        subcat_data = data[data["subcategoria"] == subcategory]
        fig.add_trace(
            go.Scatter(
                x=subcat_data["date"],
                y=subcat_data["demanda"],
                name=subcategory,
                hovertext=subcat_data["subcategoria"],
            ),
            row=i + 1,
            col=1,
        )

fig.update_layout(height=200 * len(categories), showlegend=False)
fig.show()

Con las gráficas anteriores es importante notar que el efecto de la apertura de la tienda competidora fue diferente para cada producto, subcategoría y categoría. Por ejemplo, las gráficas de aseo y producto lácteos presentan fuerte quiebres entre el 2 y el 3 de julio, mientras que los jabones y los enlatados no.

Para cuantificar el efecto de la apertura de la tienda en la demanda, me parece entonces incorrecto simplemente sumar las demandas de los diferentes productos en las fechas dadas y comparar los offsets de las demandas antes y después de la fecha, porque los productos son todos diferentes entre sí (el clásico sumar peras con manzanas), tienen precios diferentes y demandas variadas (algunos alrededor de los 500 productos diarios, y otros menos de 100), por lo que sumarlos genera un sesgo hacia los productos más demandados que puede no ser muy diciente sobre el comportamiento de demanda global para la tienda.

Por tal motivo, opto mejor por calcular las variaciones porcentuales de la demanda para cada producto por aparte, y después esos porcentajes promediarlos para estimar la variación promedio en la demanda de los diferentes productos de tal forma que todos tengan la misma importancia, evitando sesgarnos hacia la variación de los más demandados. Opto por hacerlo así pensando que en estos escenarios lo más útil para la tienda sería en realidad poder tener el precio de cada producto, y el valor de productos vendidos, con lo que sí sería bueno simplemente sumar el valor vendido de toda la tienda cada día, y evidenciar el decaimiento de esa cifra monetaria antes y después de la apertura de la tienda.

Con esto en mente, vamos a calcular entonces el (de)crecimiento porcentual de la venta de unidades por cada producto, a partir de la comparación de los promedios antes y después de la apertura de la tienda rival.

Posteriormente, se visualiza la distribución estadística de estos porcentajes, y se calcula su promedio, que corresponde a la estimación cuantitativa del impacto de la tienda competidora en ventas.


In [19]:
demanda_means = demanda.copy()
demanda_means["after_competitor"] = demanda_means["date"] > "2021-07-02"
demanda_means = (
    demanda_means.groupby(["id_producto", "after_competitor"])["demanda"]
    .mean()
    .unstack()
)

demanda_means = demanda_means.reset_index().rename(
    columns={False: "before_competitor", True: "after_competitor"}
)

demanda_means["diff_percentage"] = (
    (demanda_means["after_competitor"] - demanda_means["before_competitor"])
    / demanda_means["before_competitor"]
    * 100
)
demanda_means["cat-subcat"] = demanda_means["id_producto"].map(
    catalogo_productos["categoria"] + "-" + catalogo_productos["subcategoria"]
)

In [21]:
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Histogram(x=demanda_means["diff_percentage"]), row=1, col=1)

# Calcular promedio de la variable
average = demanda_means["diff_percentage"].mean()

# Y agregarlo como una línea vertical
fig.add_vline(
    x=average,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Promedio: {average:.2f}",
    annotation_position="top right",
)

fig.update_layout(
    title="Distribución de las diferencias porcentuales en la demanda antes y después de apertura de tienda rival, por producto",
    xaxis_title="Diferencia porcentual",
    yaxis_title="Número de productos",
    showlegend=False,
)

fig.show()

El porcentaje promedio de decremiento en ventas es del **18%**. Esto, calculando el promedio de crecimientos porcentuales para los diferentes productos.

Note que en la gráfica anterior, se observan varios productos con un decrecimiento de ventas más significativo, que supera el 40%. Veamos a qué productos corresponde:


In [70]:
demanda_means.sort_values("diff_percentage", ascending=True)[
    ["id_producto", "cat-subcat", "diff_percentage"]
].head(15)

after_competitor,id_producto,before_competitor,after_competitor,diff_percentage,cat-subcat
23,24,272.580340,115.707819,-57.550930,productos_lacteos-Leche
78,79,457.790170,200.203629,-56.267381,productos_lacteos-Desconocido
22,23,459.988395,204.395062,-55.565170,productos_lacteos-Yogur
67,68,336.577735,157.657201,-53.158755,productos_lacteos-Yogur
74,75,511.171429,239.724138,-53.102986,productos_lacteos-Queso
44,45,726.123810,341.100616,-53.024455,productos_lacteos-Desconocido
106,107,494.825758,233.391481,-52.833603,productos_lacteos-Leche
50,51,424.145299,200.247863,-52.787909,productos_lacteos-Crema
93,94,403.117534,190.845842,-52.657519,aseo-Cuidado personal
113,114,484.611969,229.879276,-52.564260,productos_lacteos-Leche


Se identifica que la categoría de **productos_lacteos** sufrió un impacto mayor al resto de productos. Veamos si existe algún sesgo similar en los productos de impacto menos negativo:


In [71]:
demanda_means.sort_values("diff_percentage", ascending=False).head(15)

after_competitor,id_producto,before_competitor,after_competitor,diff_percentage,cat-subcat
109,110,248.508637,279.830266,12.603839,productos_enlatados_y_alimentos_envasados-Desc...
20,21,301.356031,332.688623,10.397201,productos_enlatados_y_alimentos_envasados-Desc...
59,60,196.858757,215.382294,9.409557,productos_enlatados_y_alimentos_envasados-Desc...
132,133,86.514286,94.282520,8.979135,productos_enlatados_y_alimentos_envasados-Desc...
95,96,67.335878,73.153061,8.639055,productos_enlatados_y_alimentos_envasados-Desc...
102,103,184.034483,199.875776,8.607786,cereales_y_productos_secos-Frutos secos
60,61,93.872093,100.377510,6.930086,productos_enlatados_y_alimentos_envasados-Desc...
55,56,82.869482,87.250513,5.286665,productos_enlatados_y_alimentos_envasados-Desc...
127,128,223.603416,234.285421,4.777210,productos_enlatados_y_alimentos_envasados-Desc...
62,63,355.671128,372.114286,4.623135,productos_enlatados_y_alimentos_envasados-Desc...


En efecto, se observa que los productos menos afectados por la apertura de la tienda competidora son en su mayoría de la categoría **productos_enlatados_y_alimentos_envasados**, dado que estos presentan incluso leves crecimientos entre los periodos de tiempo evaluados.

Estas observaciones pueden utilizarse como insumos para estrategias de mercadeo y administrativas alrededor de los productos ofrecidos en la tienda, con respecto a la competencia.


### Tendencia de las ventas en próximos años


Se procede entonces a indagar sobre la tendencia que tendrán las ventas en la tienda durante los próximos años. Como debemos excluir de este análisis el impacto puntual de la apertura de la tienda de la competencia, considero que basta con calcular la componente lineal del comportamiento de demanda para cada producto únicamente usando los datos posteriores al 2 de julio. Esto, en aras de proponer la solución más sencilla que responda a la pregunta, y notando que afortunadamente hay un año y medio de datos posteriores al evento descrito, por lo que las afirmaciones que se extraigan de estos siguen siendo robustas.

De nuevo, sugiero hacer la regresión para cada producto por aparte y después analizar las pendientes, para evitar el sesgo hacia los productos más demandados que generaría simplemente sumar las demandas.


In [23]:
demanda_after = demanda[demanda["date"] > "2021-07-02"]

In [32]:
def fit_linear_model(demanda_after, id_producto):
    product_data = demanda_after[demanda_after["id_producto"] == id_producto].copy()

    # Garantizar que los datos estén ordenados por fecha diaria
    # esto agrega datos NaN para los días que no tenían demanda
    product_data = product_data.set_index("date").resample("D").asfreq()

    # Se agrega un identificador temporal para la regresión
    product_data["t"] = np.arange(len(product_data))

    # Podríamos intentar rellenar los potenciales datos vacíos,
    # decidí no hacerlo por lo que expliqué antes
    product_data.dropna(subset=["demanda", "t"], inplace=True)

    X = sm.add_constant(product_data["t"])
    model = OLS(product_data["demanda"], X).fit()

    # Extraigo la pendiente y su error estándar
    slope = model.params["t"]
    slope_stderr = model.bse["t"]

    return model, slope, slope_stderr

In [33]:
# Verificar que la regresión lineal se haga correctamente
model_1, slope_1, slope_1_stderr = fit_linear_model(demanda_after, 1)
product_data = demanda_after[demanda_after["id_producto"] == 1].copy()
product_data["t"] = np.arange(len(product_data))
product_data["trend"] = model_1.predict(sm.add_constant(product_data["t"]))

fig = go.Figure()

fig.add_trace(
    go.Scatter(x=product_data["t"], y=product_data["demanda"], name="Demanda real")
)
fig.add_trace(
    go.Scatter(
        x=product_data["t"],
        y=product_data["trend"],
        name="Tendencia",
        line=dict(color="red"),
    )
)

fig.update_layout(
    title="Demanda y Tendencia Lineal para Producto 1",
    xaxis_title="Identificador temporal",
    yaxis_title="Demanda",
    showlegend=True,
)

fig.show()

In [53]:
# Dataframe para guardar resultados iterativos
results = pd.DataFrame(columns=["id_producto", "slope", "stderr"])

for product_id in demanda_after["id_producto"].unique():
    model, slope, slope_stderr = fit_linear_model(demanda_after, product_id)
    results.loc[len(results.index)] = [product_id, slope, slope_stderr]

In [59]:
# unweigthed_avg_slope = results['slope'].mean()
# weighted_avg_slope = np.average(results['slope'], weights=1/results['stderr']**2)

# print(f"Pendiente promedio: {unweigthed_avg_slope:.4f}")
# print(f"Pendiente promedio ponderado: {weighted_avg_slope:.4f}")


# Pensé en hacer un análsis a través de los promedios ponderados y sin ponderar, pero creo que
# la conclusión que extraje se explica más fácilmente a través de lo siguiente que hice. Pueden
# ignorar esto.

In [54]:
results = results.merge(
    catalogo_productos[["categoria", "subcategoria"]],
    left_on="id_producto",
    right_index=True,
)
results["cat-subcat"] = results["categoria"] + "-" + results["subcategoria"]

Las pendientes se interpretan como variación de demanda por día. Para hacer más claro su análisis, hagamos un ligero post-procesamiento


In [55]:
results = results.merge(
    demanda_means[["id_producto", "after_competitor"]], on="id_producto"
)

results.rename(columns={"after_competitor": "avg_daily_demand"}, inplace=True)

results["anual_demand_variation"] = results["slope"] * 365
results["avg_anual_demand"] = results["avg_daily_demand"] * 365

results["percentual_anual_demand_variation"] = (
    results["anual_demand_variation"] / results["avg_anual_demand"] * 100
)

In [56]:
fig = go.Figure()

# Gráfico de barras con error en y
fig.add_trace(
    go.Bar(
        x=results["id_producto"],
        y=results["percentual_anual_demand_variation"],
        marker=dict(color="blue"),
        name="Variación anual percentual de demanda",
        hovertext=results["cat-subcat"],
    )
)

fig.update_layout(
    title="Variación percentual de demanda por producto",
    xaxis_title="ID producto",
    yaxis_title="Variación percentual estimada",
    showlegend=False,
)

fig.show()

Una vez más, notamos que algunos productos presentan tendencias especialmente negativas, miremos cuáles son.


In [58]:
results.sort_values("percentual_anual_demand_variation", ascending=True)[
    ["id_producto", "cat-subcat", "percentual_anual_demand_variation"]
].head(5)

,id_producto,cat-subcat,percentual_anual_demand_variation
128,69.0,bebidas-Gaseosa,-0.049712
123,15.0,cereales_y_productos_secos-Arroz,-0.048177
127,58.0,bebidas-Vinos,-0.045112
133,118.0,carnes_y_aves-Desconocido,-0.041047
125,41.0,cereales_y_productos_secos-Legumbres,-0.039690


In [60]:
len(results[results["percentual_anual_demand_variation"] < 0])

80

Aunque en efecto 80 de los 135 productos presentan una tendencia a la disminución de su demanda, incluso las cifras más críticas no superan un decrecimiento superior al 0.05% anual, que me parece despreciable y propenso a error estadístico y de la muestra o método usado. Por tanto, mi conclusión frente a este ítem sería indicar que la información indica que la demanda general se mantendrá estable en los próximos años, pero que amerita seguir atentos al comportamiento de estos estimadores para ver si la tendencía se vuelve definidamente negativa.


Una vez obtenidas las conclusiones hacia las preguntas planteadas, prosigo con el planteamiento de los modelos de pronóstico.


## Modelos para predicción


### Base model


El primer paso para la proposición del modelo es la generación de un modelo base que solucione el problema en cuestión, y con el cuál se puedan comparar modelos posteriores para evaluarlos.

La estrategia propuesta para esto es elemental: dado que encontramos que la tendencia de los productos es aproximadamente constante, y que estos presentan periodicidad anual, se propone como modelo base que, para un día dado, se prediga la demanda como la misma demanda que hubo un año atrás.

Evaluemos el RMSE de esta propuesta para aproximadamente los últimos 6 meses de datos de demanda suministrados.


In [ ]:
# tomé esta fecha porque divide los datos de demanda disponibles aproximandamente en
# 2.5 años para entrenar y 6 meses para hacer test, que me parece un buen horizonte
# comparado con el año que hay que predecir para la prueba
TRAIN_TEST_CUTOFF_DATE = "2022-06-01"


def root_mean_squared_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [203]:
def get_base_prediction(demanda: pd.DataFrame, product_id: int, cutoff_date: str):
    """
    Calcula la predicción base para un producto en particular
    """
    df = demanda[demanda["id_producto"] == product_id].copy()
    df_test = df[df["date"] > cutoff_date].copy()

    predicted_values = []

    for date in df_test["date"]:
        date_prev_year = date - pd.DateOffset(years=1)

        predicted_value = []
        while len(predicted_value) == 0:
            predicted_value = df[df["date"] == date_prev_year]["demanda"].values
            date_prev_year -= pd.DateOffset(days=1)

        predicted_value = predicted_value[0]
        predicted_values.append(predicted_value)

    df_test["yhat"] = predicted_values

    return df_test[["date", "yhat", "demanda"]]

In [204]:
product_id = 1
df = demanda[demanda["id_producto"] == product_id].copy()
df_test = get_base_prediction(demanda, product_id, TRAIN_TEST_CUTOFF_DATE)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["demanda"], name="Demanda real"))
fig.add_trace(
    go.Scatter(
        x=df_test["date"],
        y=df_test["yhat"],
        name="Demanda predicha",
        line=dict(color="red"),
    )
)

fig.update_layout(
    title=f"Demanda real y predicha para Producto {product_id}",
    xaxis_title="Fecha",
    yaxis_title="Demanda",
    showlegend=True,
)

fig.show()

rmse = root_mean_squared_error(df_test["demanda"], df_test["yhat"])
print(f"El RMSE para estos datos es de: {rmse}")

El RMSE para estos datos es de: 18.121548845073335


No funciona para nada mal! Ahora, veamos el RMSE resultante cuando se aplica esto para todos los productos, del 1 de junio de 2022 en adelante.

Veamos también como se comporta ese RMSE para los diferentes productos, con el fin de seguir verificando la variedad de comportamientos que presentan los productos


In [261]:
predicted = []
base_rmses = []

for product_id in demanda["id_producto"].unique():
    df_test = get_base_prediction(demanda, product_id, TRAIN_TEST_CUTOFF_DATE)
    predicted.append(df_test)

    rmse = root_mean_squared_error(df_test["demanda"], df_test["yhat"])
    base_rmses.append(rmse)

predicted = pd.concat(predicted)
rmse = root_mean_squared_error(predicted["demanda"], predicted["yhat"])

print(f"El RMSE para todos los productos es de: {rmse}")

El RMSE para todos los productos es de: 78.01418781162032


In [262]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=demanda["id_producto"].unique(), y=base_rmses, mode="markers")
)

fig.update_layout(
    title="RMSE por producto",
    xaxis_title="ID Producto",
    yaxis_title="RMSE",
    showlegend=False,
)

fig.show()

### Prophet: un modelo por producto

Una vez tomadas medidas de referencia para el RMSE, es hora de proponer modelos más robustos de forecasting.

Inicialmente creo que es mejor hacer un modelo para cada producto, ya que hay 2.5 años de datos (~1000 datos por producto), y creo que esto no es suficiente para hacer un modelo que prediga para todos los productos a la vez.

-   Hacer la predicción para la demanda de cada producto por aparte, usando el modelo **Prophet** de Meta Open Source. Esto debido a:
    -   El análisis gráfico hecho previamente de las series de tiempo me hace pensar que usar modelos enfocados en cada producto por aparte puede ser más sencillo, directo y garantizar mejores resultados
    -   Experiencias previas con Prophet muy positivas
    -   Practicidad en la solución, proponer la solución más sencilla que resuelva el problema.

Sin embargo, reconozco que esta estrategia no haría uso de las características de cada producto que me fueron dadas en el catálogo, ni de la posibilidad de apoyar el modelo de la información previa no solo del mismo producto, sino de otros productos o estimadores relacionados, que potencialmente podría mejorar los resultados

Por tanto, una segunda estrategia es:

-   Armar un vector de entrada manualmente, que contenga los valores anteriores de todos los productos,

TODO


In [1]:
# creé algunas funciones durante la refactorización del código para
# mantenerlo un poco más DRY y auto-documentado
def get_df_prophet_ready(demanda, product_id):
    df = demanda[demanda["id_producto"] == product_id].copy()
    df = df.rename(columns={"date": "ds", "demanda": "y"})
    return df

In [268]:
# probemos una primera implementación de prophet
product_id = 1
df = get_df_prophet_ready(demanda, product_id)

model = Prophet()

# ajustemos con todos los datos hasta el 1 de junio de 2022
df_train = df[df["ds"] < TRAIN_TEST_CUTOFF_DATE].copy()
df_test = df[df["ds"] >= TRAIN_TEST_CUTOFF_DATE].copy()
model.fit(df_train)

forecast = model.predict(df)

fig = go.Figure()

fig.add_trace(
    go.Scatter(x=df_train["ds"], y=df_train["y"], mode="markers", name="Demand (train)")
)
fig.add_trace(
    go.Scatter(x=df_test["ds"], y=df_test["y"], mode="markers", name="Demand (test)")
)
fig.add_trace(
    go.Scatter(x=forecast["ds"], y=forecast["yhat"], mode="lines", name="Forecasted")
)
fig.show()

forecast_tested = forecast[forecast["ds"] >= TRAIN_TEST_CUTOFF_DATE]
rmse = root_mean_squared_error(df_test["y"], forecast_tested["yhat"])

print(f"El RMSE para este producto, con el modelo por defecto de Prophet es de: {rmse}")

22:07:52 - cmdstanpy - INFO - Chain [1] start processing
22:07:52 - cmdstanpy - INFO - Chain [1] done processing


El RMSE para este producto, con el modelo por defecto de Prophet es de: 51.74309675320552


In [270]:
future = model.make_future_dataframe(periods=180)

forecast = model.predict(future)

fig = plot_plotly(model, forecast)
fig.show()

Vemos que el RMSE para este producto con el modelo por defecto da incluso peor que el base. Resulta pertinente entonces ajustar los hiperparámetros para mejorar los resultados. El modelo Prophet trabaja modelando la serie de tiempo con 3 términos, uno para la tendencia lineal y no-lineal en el tiempo, otro para las estacionalidades (periodicidad), y otro para otros regresores. Es un modelo pensado para aplicación rápida, práctica y con amplia usabilidad en predicción de series de tiempo para temas comerciales.

En este marco, es posible indicarle específicamente que estacionalidades vemos en los productos y que peso inicial le damos a esas diferentes estacionalidades, así como por ejemplo agregarle regresores adicionales como los días festivos y también modificar sus pesos adicionales.

Para plasmar la metodología de ajuste de hiperparámetros, se presenta el siguiente código en el que se hace un mallado de parámetros iniciales y se aplica una búsqueda tipo Grid Search, es decir, se intentan todas las combinaciones y se elige la mejor con base a una métrica.

Además, se plasma el uso de validación cruzada para evitar sesgos por la selección de los subconjuntos de entrenamiento y prueba. Por ser datos temporales, no es posible hacer una validación cruzada tradicional, sino que esta garantiza la creación de diferentes ventanas en los datos de tal forma que el entrenamiento siempre está antes que la prueba, y que se mantienen continuos en su temporalidad y el de entrenamiento con datos más antiguos que la prueba. En este caso se utilizan ventanas de 540 para entrenamiento, para la prueba en horizontes de 180 días, y periodos de corte entre las ventanas de 150 días. Esto permite generar 3 validaciones cruzadas diferentes en el set de datos facilitado.

Igual, la malla a explorar está cargada del conocmienot previo del ejercicio. Notamos los picos incrementales en la data para festivos en Colombia, por lo que agregamos esta variable como regresora y exploramos su peso. Conocemos la influencia periodica del sistema, por lo que buscamos también en pesos altos, y por la misma razón mantenemos los de la tendencia bajos.

Para no complicar demasiado más el ejercicio, no voy a hacer este ajuste para cada uno de los modelos de cada uno de los productos, dado que sabemos de antemano que tienen comportamientos similares le permitiremos al modelo maniobrar su predicción con los mismos hiperparámetros


In [352]:
product_id = 1
df = get_df_prophet_ready(demanda, product_id)

param_grid = {
    "changepoint_prior_scale": [0.01, 0.05, 0.1],
    "seasonality_prior_scale": [1.0, 10.0, 20.0],
    "holidays_prior_scale": [1.0, 10.0, 20.0],
    "seasonality_mode": ["additive", "multiplicative"],
    "weekly_seasonality": [True],
    "daily_seasonality": [False],
    "yearly_seasonality": [True],
}

# Genera diccionarios con todas las combinaciones de parámetros
all_params = [
    dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())
]
rmses = []

# Usar validación cruzada para evaluar parámetros
for params in tqdm(all_params, desc="Combinación de hiperparámetros "):
    m = Prophet(**params)
    m.add_country_holidays(country_name="CO")
    m.fit(df)  # Fit model with given params
    df_cv = cross_validation(
        m,
        initial="540 days",
        period="150 days",
        horizon="180 days",
        parallel="processes",
    )
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses.append(df_p["rmse"].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results["rmse"] = rmses

Combinación de hiperparámetros : 100%|██████████| 54/54 [04:24<00:00,  4.89s/it]


In [353]:
tuning_results.sort_values("rmse").head(10)

,changepoint_prior_scale,seasonality_prior_scale,holidays_prior_scale,seasonality_mode,weekly_seasonality,daily_seasonality,yearly_seasonality,rmse
29,0.05,10.0,20.0,multiplicative,True,False,True,15.089526
35,0.05,20.0,20.0,multiplicative,True,False,True,15.402430
25,0.05,10.0,1.0,multiplicative,True,False,True,15.418563
19,0.05,1.0,1.0,multiplicative,True,False,True,15.431942
33,0.05,20.0,10.0,multiplicative,True,False,True,15.481386
27,0.05,10.0,10.0,multiplicative,True,False,True,15.548284
23,0.05,1.0,20.0,multiplicative,True,False,True,15.584380
31,0.05,20.0,1.0,multiplicative,True,False,True,15.670124
21,0.05,1.0,10.0,multiplicative,True,False,True,15.720313
18,0.05,1.0,1.0,additive,True,False,True,16.459546


Note aquí otra ventaja del modelo Prophet, y es que no resulta al final tan necesario hacer esta búsqueda de hiperparámetros tan exhaustiva, puesto que nuestra comprensión de la data ya nos permite estimar valores óptimos con alta certeza. La variable **changepoint_prior_scale** hace referencia a la influencia que se le da a la parte lineal (y no-lineal, eventos discretos) en la serie de tiempo, pero esto de antemano sabíamos que era pequeño, por lo que tiene sentido que los mejores resultados tengan este valor bajo.

Veamos gráficamente cómo se ven esos valores


In [358]:
def best_prophet():
    """Función que genera el model de Prophet con los mejores hiperparámetros"""
    model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
        holidays_prior_scale=20,
        seasonality_prior_scale=10.0,
        changepoint_prior_scale=0.05,
        seasonality_mode="multiplicative",
    )
    model.add_country_holidays(country_name="CO")
    return model

In [354]:
product_id = 1

df = get_df_prophet_ready(demanda, product_id)

model = best_prophet()
model.fit(df)

forecast = model.predict(df)
plot_components_plotly(model, forecast)

En la gráfica anterior se notan los comportamientos temporales de la información, se observa la importancia de tener en cuenta los días festivos, y se percibe el efecto en el día de la semana y el año.

Todo esto corresponde a las conclusiones que se extraerían en un ejercicio tradicional de ingeniería de datos, solo que aquí el modelo incluso nos automatizó su extracción y uso para la predicción.


Veamos que en efecto con este modelo el RMSE para todos los productos mejore, y sobre todo su comportamiento con los productos estacionales, que puede que de problemas adicionales.


In [355]:
def get_best_prophet_prediction(demanda, product_id):
    df = get_df_prophet_ready(demanda, product_id)

    model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
        holidays_prior_scale=20,
        seasonality_prior_scale=10.0,
        changepoint_prior_scale=0.05,
        seasonality_mode="multiplicative",
    )
    model.add_country_holidays(country_name="CO")

    # ajustemos con todos los datos hasta el 1 de junio de 2022
    df_train = df[df["ds"] < "2022-06-01"].copy()
    df_test = df[df["ds"] >= "2022-06-01"].copy()
    model.fit(df_train)

    forecast = model.predict(df)
    forecast_tested = forecast[forecast["ds"] >= "2022-06-01"]
    df_test["predicted"] = forecast_tested["yhat"].values

    return df_test

In [356]:
predicted = []
prophet_rmses = []

for product_id in tqdm(demanda["id_producto"].unique(), desc="Producto "):
    df_test = get_best_prophet_prediction(demanda, product_id)
    predicted.append(df_test)

    rmse = root_mean_squared_error(df_test["y"], df_test["predicted"])
    prophet_rmses.append(rmse)

predicted = pd.concat(predicted)
rmse = root_mean_squared_error(predicted["y"], predicted["predicted"])

print(f"El RMSE para todos los productos es de: {rmse}")

Producto : 100%|██████████| 135/135 [01:51<00:00,  1.21it/s]

El RMSE para todos los productos es de: 37.44513743233536


In [357]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=demanda["id_producto"].unique(),
        y=prophet_rmses,
        mode="markers",
        name="RMSE Prophet tuned",
    )
)
fig.add_trace(
    go.Scatter(
        x=demanda["id_producto"].unique(),
        y=base_rmses,
        mode="markers",
        name="RMSE base model",
    )
)

fig.update_layout(
    title="RMSE por producto",
    xaxis_title="ID Producto",
    yaxis_title="RMSE",
    showlegend=True,
)
fig.show()

### Completar el dataset demanda_test

Antes de completar el dataset a entregar para la prueba, voy a verificar cómo se está comportando el modelo estimador para los productos estacionales


In [271]:
product_id = 4  # uno de los productos estacionales

df = get_df_prophet_ready(demanda, product_id)
model = best_prophet()
model.fit(df)

future = model.make_future_dataframe(periods=180)  # que prediga los siguientes 180 días
forecast = model.predict(future)

22:13:57 - cmdstanpy - INFO - Chain [1] start processing
22:13:57 - cmdstanpy - INFO - Chain [1] done processing


In [272]:
plot_plotly(model, forecast)

Como sospechaba, igual el modelo está alucinando en estos casos para las fechas que no aparecen como parte de la serie de tiempo para estos escenarios. Sería bueno explorar tal vez la generación de otro modelo diferente que fuera robusto específicamente a estos escenarios, o haber considerado rellenar la demanda de estos productos particulares como nula para los meses en los que no se venden, pero no quisiera seguir reiterando en partes anteriores del notebook.

Ante esta situación, lo más práctico y sensato es que verifique que estas fechas no aparezcan en el dataset de demanda_test para los productos estacionales.


In [336]:
stationary_product_ids = catalogo_productos[
    catalogo_productos["estacional"] == 1
].index.values
zero_demand_stationary_months = [2, 3, 4, 5, 6, 7, 8, 9, 10]

demanda_stationary_mask = (demanda_test["id_producto"].isin(stationary_product_ids)) & (
    demanda_test["date"].dt.month.isin(zero_demand_stationary_months)
)

len(demanda_test[demanda_stationary_mask])

0

In [273]:
# perfecto, no hay fechas de este estilo, entonces procedo a completar el dataset con las predicciones
results = []
forecast_frames = []

for product_id in demanda["id_producto"].unique():
    df = get_df_prophet_ready(demanda, product_id)
    model = best_prophet()
    model.fit(df)

    future_dates = demanda_test[demanda_test["id_producto"] == product_id][
        "date"
    ].unique()
    future = pd.DataFrame(future_dates, columns=["ds"])
    forecast = model.predict(future)
    forecast["id_producto"] = product_id
    forecast_frames.append(forecast[["ds", "id_producto", "yhat"]])

final_forecasts = pd.concat(forecast_frames)

22:18:41 - cmdstanpy - INFO - Chain [1] start processing
22:18:42 - cmdstanpy - INFO - Chain [1] done processing
22:18:42 - cmdstanpy - INFO - Chain [1] start processing
22:18:42 - cmdstanpy - INFO - Chain [1] done processing
22:18:42 - cmdstanpy - INFO - Chain [1] start processing
22:18:43 - cmdstanpy - INFO - Chain [1] done processing
22:18:43 - cmdstanpy - INFO - Chain [1] start processing
22:18:44 - cmdstanpy - INFO - Chain [1] done processing
22:18:44 - cmdstanpy - INFO - Chain [1] start processing
22:18:44 - cmdstanpy - INFO - Chain [1] done processing
22:18:45 - cmdstanpy - INFO - Chain [1] start processing
22:18:45 - cmdstanpy - INFO - Chain [1] done processing
22:18:45 - cmdstanpy - INFO - Chain [1] start processing
22:18:45 - cmdstanpy - INFO - Chain [1] done processing
22:18:46 - cmdstanpy - INFO - Chain [1] start processing
22:18:46 - cmdstanpy - INFO - Chain [1] done processing
22:18:47 - cmdstanpy - INFO - Chain [1] start processing
22:18:47 - cmdstanpy - INFO - Chain [1]

In [277]:
final_forecasts.rename(columns={"ds": "date", "yhat": "demanda"}, inplace=True)

# así garantizo que el orden de demanda_test se mantenga,
# para evitar problemas en el algoritmo de evaluación
demanda_test = demanda_test.merge(
    final_forecasts, on=["id_producto", "date"], how="left"
)

# convertir a entero para que tenga sentido la variable para el cliente.
demanda_test["demanda"] = demanda_test["demanda"].astype(int)
demanda_test.to_csv("demanda_test_filled.csv", index=False)